# TP Sismique Réflexion

Exécutez chaque cellule avec `Shift+Enter`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve
%matplotlib inline

In [ ]:
# Ondelette de Ricker
dt = 0.004
t = np.arange(-0.1, 0.1, dt)
f0 = 25
wavelet = (1 - 2*(np.pi*f0*t)**2) * np.exp(-(np.pi*f0*t)**2)

plt.figure(figsize=(10,4))
plt.plot(t, wavelet)
plt.title('Ondelette de Ricker (25 Hz)')
plt.grid(True)
plt.show()

In [ ]:
# Modèle de vitesse
z = np.linspace(0, 2000, 500)
v = 1500 + 0.5 * z
v[z > 1500] = 3500

plt.figure(figsize=(8,6))
plt.plot(v, z)
plt.xlabel('Vitesse (m/s)')
plt.ylabel('Profondeur (m)')
plt.title('Modèle de vitesse')
plt.gca().invert_yaxis()
plt.grid(True)
plt.show()

In [ ]:
# Trace synthétique
time = np.arange(0, 2, dt)
reflectors = [(0.2, 0.3), (0.5, -0.2), (0.9, 0.4), (1.4, -0.1)]

reflectivity = np.zeros_like(time)
for t0, amp in reflectors:
    idx = np.argmin(np.abs(time - t0))
    reflectivity[idx] = amp

trace = convolve(reflectivity, wavelet, mode='same')

plt.figure(figsize=(12,6))
plt.subplot(2,1,1)
plt.stem(time, reflectivity, basefmt=" ")
plt.title('Réflectivité')
plt.subplot(2,1,2)
plt.plot(time, trace)
plt.title('Trace sismique')
plt.xlabel('Temps (s)')
plt.tight_layout()
plt.show()

In [ ]:
# Correction NMO
offsets = np.arange(100, 2000, 200)
v_rms = 1800

gather = []
for offset in offsets:
    trace = np.zeros_like(time)
    for t0, amp in reflectors:
        t = np.sqrt(t0**2 + (offset/v_rms)**2)
        idx = np.argmin(np.abs(time - t))
        if idx < len(trace):
            trace[idx] = amp
    trace = convolve(trace, wavelet, mode='same')
    gather.append(trace)

gather = np.array(gather).T

plt.figure(figsize=(10,8))
plt.imshow(gather, aspect='auto', cmap='seismic',
           extent=[offsets[0], offsets[-1], time[-1], 0])
plt.colorbar(label='Amplitude')
plt.xlabel('Offset (m)')
plt.ylabel('Temps (s)')
plt.title('Gather avant correction NMO')
plt.show()

In [ ]:
# Empilement
stack = np.mean(gather, axis=1)

plt.figure(figsize=(10,6))
plt.plot(stack, time, 'k-', linewidth=1.5)
plt.ylim(time[-1], 0)
plt.xlabel('Amplitude')
plt.ylabel('Temps (s)')
plt.title('Trace empilée')
plt.grid(True)
plt.show()

# Félicitations !